# A. CONFIG
**Required, under 1 minute.** Edit this cell only. It defines every path, run name, shared numerical setting, confirmation, and monitor action used below.

In [ ]:
from pathlib import Path
import json, subprocess, sys

WORKSPACE_ROOT = Path("/workspace")
REPO_ROOT = WORKSPACE_ROOT / "TRDN-Video-Dehazing"
DATASET_ROOT = WORKSPACE_ROOT / "datasets" / "REVIDE"
SESSION_ROOT = WORKSPACE_ROOT / "trdn_paper_run"
RUNS_ROOT = SESSION_ROOT / "runs"
REPORTS_ROOT = SESSION_ROOT / "reports"
EVAL_DIR = SESSION_ROOT / "evaluations"
ARTIFACTS_DIR = SESSION_ROOT / "paper_artifacts"
PRESET_PATH = REPO_ROOT / "configs" / "a40.yaml"
ENV_JSON = REPORTS_ROOT / "environment.json"
DATASET_JSON = REPORTS_ROOT / "dataset_check.json"
VAE_JSON = REPORTS_ROOT / "vae_ceiling.json"
PREFLIGHT_ROOT = SESSION_ROOT / "preflight"
PREFLIGHT_JSON = PREFLIGHT_ROOT / "preflight_report.json"
BENCHMARK_JSON = REPORTS_ROOT / "benchmark_report.json"
SHARED_SAMPLES_JSON = ARTIFACTS_DIR / "shared_sample_selection.json"
BUNDLE_PATH = WORKSPACE_ROOT / "TRDN_paper_run_bundle.zip"
RUN_NAMES = {name: name for name in ("full", "no_raft", "no_transformer", "diffusion_only")}
SCRIPTS = {name: REPO_ROOT / "scripts" / name for name in (
    "runpod_workflow.py", "runpod_jobs.py", "vae_ceiling.py", "preflight.py",
    "benchmark.py", "make_paper_figures.py", "emit_paper_tables.py"
)}

SEED = 1234
SEQ_LEN = 10
CROP_SIZE = 256
NUM_EPOCHS = 30
MAX_TRAIN_STEPS = 0
NUM_WORKERS = 4
INFERENCE_STEPS = 50
VALIDATION_NUM_SAMPLES = 32
VALIDATION_NUM_STEPS = 30
CHECKPOINT_SELECTION_METRIC = "psnr"
NUM_QUALITATIVE_SAMPLES = 4
NUM_FAILURE_CLIPS = 3
BENCHMARK_WARMUP_STEPS = 3
BENCHMARK_TIMED_STEPS = 10
BENCHMARK_MAX_BATCH_SIZE = 32
FORCE = False
INSTALL_DEPENDENCIES = True
NUMERICS_CONFIRMATION = ""  # Set to LOCK_A40 only after reviewing cell G.
NUMERICS_OVERRIDES = {}  # Optional keys from configs/a40.yaml.
MONITOR_ACTION = "list"  # list, monitor, follow, or stop
MONITOR_VARIANT = "full"

for path in (SESSION_ROOT, RUNS_ROOT, REPORTS_ROOT, EVAL_DIR, ARTIFACTS_DIR):
    path.mkdir(parents=True, exist_ok=True)

def call_script(name, *arguments):
    command = [sys.executable, str(SCRIPTS[name]), *map(str, arguments)]
    print("$", " ".join(command), flush=True)
    return subprocess.run(command, cwd=REPO_ROOT, check=True)

def maybe_force(arguments):
    return [*arguments, "--force"] if FORCE else list(arguments)

def launch_variant(variant):
    arguments = [
        "launch-training", "--variant", variant, "--run-name", RUN_NAMES[variant],
        "--project-root", RUNS_ROOT / variant, "--runs-root", RUNS_ROOT,
        "--dataset-root", DATASET_ROOT, "--preset", PRESET_PATH, "--seed", SEED,
        "--num-epochs", NUM_EPOCHS, "--max-train-steps", MAX_TRAIN_STEPS,
        "--num-workers", NUM_WORKERS, "--seq-len", SEQ_LEN, "--crop-size", CROP_SIZE,
        "--validation-num-samples", VALIDATION_NUM_SAMPLES,
        "--validation-num-steps", VALIDATION_NUM_STEPS,
        "--checkpoint-selection-metric", CHECKPOINT_SELECTION_METRIC,
        "--resume-if-interrupted",
    ]
    call_script("runpod_workflow.py", *maybe_force(arguments))


# B. ENVIRONMENT
**Required, roughly 5-15 minutes on first run.** Installs dependencies when enabled, records the GPU, VRAM, and package versions, and fails if CUDA is unavailable.

In [ ]:
arguments = ["environment", "--output", ENV_JSON]
if INSTALL_DEPENDENCIES:
    arguments.append("--install")
call_script("runpod_workflow.py", *maybe_force(arguments))


# C. DATASET CHECK
**Required, roughly 1-10 minutes.** Validates train/validation/test structure, natural frame pairing, sequence and frame counts before model construction.

In [ ]:
call_script("runpod_workflow.py", *maybe_force([
    "dataset-check", "--dataset-root", DATASET_ROOT, "--output", DATASET_JSON,
    "--seed", SEED, "--seq-len", SEQ_LEN, "--crop-size", CROP_SIZE,
]))


# D. VAE CEILING
**Required, potentially tens of minutes.** Measures the deterministic VAE round-trip ceiling. This is the maximum achievable PSNR/SSIM/LPIPS at the configured resolution; every trained result is bounded by it.

In [ ]:
if VAE_JSON.is_file() and not FORCE:
    print(f"SKIP: {VAE_JSON} exists")
else:
    call_script("vae_ceiling.py", "--dataset-root", DATASET_ROOT, "--resolution", CROP_SIZE, "--output", VAE_JSON)
print(json.dumps(json.loads(VAE_JSON.read_text()), indent=2))


# E. PREFLIGHT
**Required, roughly 10-30 minutes.** Runs dataset integrity, loss, determinism, measured timing, and measured checkpoint-storage checks for the full model.

In [ ]:
if PREFLIGHT_JSON.is_file() and not FORCE:
    print(f"SKIP: {PREFLIGHT_JSON} exists")
else:
    call_script("preflight.py", "--dataset-root", DATASET_ROOT, "--project-root", PREFLIGHT_ROOT,
                "--num-epochs", NUM_EPOCHS, "--max-train-steps", MAX_TRAIN_STEPS,
                "--seed", SEED, "--seq-len", SEQ_LEN, "--crop-size", CROP_SIZE,
                "--num-workers", NUM_WORKERS, "--validation-num-samples", VALIDATION_NUM_SAMPLES,
                "--validation-num-steps", VALIDATION_NUM_STEPS,
                "--checkpoint-selection-metric", CHECKPOINT_SELECTION_METRIC)
print(json.dumps(json.loads(PREFLIGHT_JSON.read_text()), indent=2))


# F. BENCHMARK
**Required, potentially 1-3 hours.** Measures every A40 option independently, prints its table, records numerics changes, and derives a measured preset recommendation and four-run projection.

In [ ]:
if BENCHMARK_JSON.is_file() and not FORCE:
    print(f"SKIP: {BENCHMARK_JSON} exists")
else:
    call_script("benchmark.py", "--dataset-root", DATASET_ROOT, "--project-root", SESSION_ROOT / "benchmark",
                "--output", BENCHMARK_JSON, "--warmup-steps", BENCHMARK_WARMUP_STEPS,
                "--timed-steps", BENCHMARK_TIMED_STEPS, "--max-batch-size", BENCHMARK_MAX_BATCH_SIZE,
                "--seed", SEED, "--planned-epochs", NUM_EPOCHS, "--planned-runs", len(RUN_NAMES))
print(json.dumps(json.loads(BENCHMARK_JSON.read_text()), indent=2))


# G. GATE: STOP BEFORE TRAINING
**Required review, no compute.** Stop here. Review the VAE ceiling, preflight, benchmark recommendation, projected wall-clock, and projected checkpoint storage. Decide whether to reduce scope before spending on four training runs.

In [ ]:
call_script("runpod_workflow.py", "gate", "--vae-json", VAE_JSON,
            "--preflight-json", PREFLIGHT_JSON, "--benchmark-json", BENCHMARK_JSON)


# H. LOCK NUMERICS
**Required, under 1 minute.** After review, set `NUMERICS_CONFIRMATION` in cell A. This writes and validates the A40 preset from measured recommendations plus explicit overrides; training refuses unresolved TODOs.

In [ ]:
call_script("runpod_workflow.py", "lock-numerics", "--benchmark-json", BENCHMARK_JSON,
            "--output", PRESET_PATH, "--confirm", NUMERICS_CONFIRMATION,
            "--overrides-json", json.dumps(NUMERICS_OVERRIDES))


# I. TRAINING RUN 1: FULL MODEL
**Required, many hours.** Launches the full model as a detached process. The cell returns immediately; browser or kernel disconnects do not terminate training.

In [ ]:
launch_variant("full")


# J. MONITOR, LIST, OR STOP
**Optional and re-runnable, seconds unless follow mode is used.** Set `MONITOR_ACTION` and `MONITOR_VARIANT` in cell A. Listing and one-shot monitoring never affect jobs; stop sends a clean termination request.

In [ ]:
call_script("runpod_jobs.py", "list", "--runs-root", RUNS_ROOT)
monitor_dir = RUNS_ROOT / MONITOR_VARIANT / "logs" / "runs" / RUN_NAMES[MONITOR_VARIANT]
if MONITOR_ACTION in {"monitor", "follow"}:
    arguments = ["monitor", "--run-dir", monitor_dir]
    if MONITOR_ACTION == "follow":
        arguments.append("--follow")
    call_script("runpod_jobs.py", *arguments)
elif MONITOR_ACTION == "stop":
    call_script("runpod_jobs.py", "stop", "--run-dir", monitor_dir)
elif MONITOR_ACTION != "list":
    raise ValueError("MONITOR_ACTION must be list, monitor, follow, or stop")


# K. TRAINING RUN 2: NO RAFT
**Required, many hours.** Launches the no-RAFT training ablation detached, with the same locked numerics and seed.

In [ ]:
launch_variant("no_raft")


# L. TRAINING RUN 3: NO TRANSFORMER
**Required, many hours.** Launches the no-transformer training ablation detached, with the same locked numerics and seed.

In [ ]:
launch_variant("no_transformer")


# M. TRAINING RUN 4: DIFFUSION ONLY
**Required, many hours.** Launches the separately trained U-Net-only baseline detached. Temporal modules are not constructed and only diffusion, L1, and LPIPS losses are active.

In [ ]:
launch_variant("diffusion_only")


# N. EVALUATE ALL RUNS
**Required after all jobs complete, potentially hours.** Evaluates each selected checkpoint over the full test set and prints found, evaluated, and skipped sample accounting. Metric RAFT is evaluator-owned for every variant.

In [ ]:
call_script("runpod_workflow.py", *maybe_force([
    "evaluate-all", "--runs-root", RUNS_ROOT, "--dataset-root", DATASET_ROOT,
    "--eval-dir", EVAL_DIR, "--preset", PRESET_PATH, "--num-steps", INFERENCE_STEPS,
    "--seed", SEED, "--seq-len", SEQ_LEN, "--crop-size", CROP_SIZE,
]))


# O. FINAL PAPER FIGURES
**Required, potentially hours.** Creates the complete deterministic PNG/PDF figure set. Shared sample indices are written once before drawing and reused across all runs; the final checklist fails on any missing figure.

In [ ]:
eval_jsons = [EVAL_DIR / f"{variant}.json" for variant in RUN_NAMES]
metric_logs = [RUNS_ROOT / variant / "logs" / "runs" / RUN_NAMES[variant] / "metrics.jsonl" for variant in RUN_NAMES]
full_checkpoint = RUNS_ROOT / "full" / "checkpoints" / f"best_{CHECKPOINT_SELECTION_METRIC}"
call_script("make_paper_figures.py", *maybe_force([
    "--checkpoint", full_checkpoint, "--eval-json", *eval_jsons,
    "--dataset-root", DATASET_ROOT, "--output-dir", ARTIFACTS_DIR,
    "--seed", SEED, "--num-samples", NUM_QUALITATIVE_SAMPLES,
    "--shared-samples-json", SHARED_SAMPLES_JSON, "--metric-log", *metric_logs,
    "--vae-ceiling-json", VAE_JSON, "--num-failure-clips", NUM_FAILURE_CLIPS,
]))


# P. PAPER TABLES
**Required, under 1 minute.** Reads only the four evaluation JSONs, writes Markdown and CSV tables, and displays the Markdown inline.

In [ ]:
table_markdown = ARTIFACTS_DIR / "paper_results.md"
table_csv = ARTIFACTS_DIR / "paper_results.csv"
if table_markdown.is_file() and table_csv.is_file() and not FORCE:
    print(f"SKIP: paper tables exist in {ARTIFACTS_DIR}")
else:
    call_script("emit_paper_tables.py", "--eval-json", *[EVAL_DIR / f"{variant}.json" for variant in RUN_NAMES],
                "--output-dir", ARTIFACTS_DIR, "--basename", "paper_results")
from IPython.display import Markdown, display
display(Markdown(table_markdown.read_text()))


# Q. DOWNLOAD BUNDLE
**Required, a few minutes.** Verifies and archives manifests, metric logs, evaluation JSONs, all figures and sidecars, shared samples, tables, and the locked preset. This archive is what leaves the pod.

In [ ]:
call_script("runpod_workflow.py", *maybe_force([
    "bundle", "--workspace-root", WORKSPACE_ROOT, "--runs-root", RUNS_ROOT,
    "--eval-dir", EVAL_DIR, "--artifacts-dir", ARTIFACTS_DIR,
    "--preset", PRESET_PATH, "--output", BUNDLE_PATH,
]))
